In [4]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_excel(r"C:\Users\ODAMA\Downloads\04_ZenithBank_Financial_Report.xlsx",
                   sheet_name='Financial Data',
                   header=2)

# Recalculate formula columns
df['Total Revenue (₦)'] = df['Interest Income (₦)'] + df['Non-Interest Income (₦)']
df['Net Profit (₦)']    = df['Total Revenue (₦)'] - df['Operating Expenses (₦)'] - df['Loan Loss Provision (₦)']

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_revenue = df['Total Revenue (₦)'].sum()
total_profit  = df['Net Profit (₦)'].sum()
avg_roe       = df['Return on Equity %'].mean()
avg_npl       = df['NPL Ratio %'].mean()

# ── CHART DATA ────────────────────────────────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)
df = df.sort_values(['Year','Month']).reset_index(drop=True)
df['Month-Year'] = df['Month'].astype(str) + ' ' + df['Year'].astype(str)

quarterly = df.groupby(['Year','Quarter'])[['Total Revenue (₦)','Operating Expenses (₦)']].sum().reset_index()
quarterly['Period'] = quarterly['Quarter'].astype(str) + ' ' + quarterly['Year'].astype(str)

# ── CHART STYLE HELPER ────────────────────────────────────────────────────────
def style(fig):
    fig.update_layout(
        plot_bgcolor='#ffffff',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial', size=13),
        title_font=dict(size=16, color='#1f3b5c', family='Arial'),
        margin=dict(l=20, r=20, t=50, b=80),
        height=380,
        showlegend=True,
        legend=dict(
            bgcolor='rgba(255,255,255,0.95)',
            bordercolor='#d6e6f2',
            borderwidth=1,
            font=dict(size=12, color='#1f3b5c'),
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        xaxis=dict(tickangle=45, tickfont=dict(size=11), title_font=dict(size=13), showgrid=True, gridcolor='#f0f4f8'),
        yaxis=dict(tickfont=dict(size=11), title_font=dict(size=13), showgrid=True, gridcolor='#f0f4f8')
    )
    return fig

# ── CHARTS ────────────────────────────────────────────────────────────────────

# 1. Revenue Trend
fig_revenue = px.line(df, x='Month-Year', y='Total Revenue (₦)',
                      markers=True, title='Total Revenue Trend')
fig_revenue.update_traces(line_color='#1f6fbf', line_width=3,
                           marker=dict(color='#1f6fbf', size=8,
                                       line=dict(width=1.5, color='white')))
style(fig_revenue)

# 2. Net Profit
fig_profit = px.bar(df, x='Month-Year', y='Net Profit (₦)',
                    color='Net Profit (₦)', color_continuous_scale='Blues',
                    title='Net Profit by Month')
style(fig_profit)

# 3. Quarterly Revenue vs Expenses
fig_quarterly = px.bar(quarterly, x='Period',
                       y=['Total Revenue (₦)', 'Operating Expenses (₦)'],
                       barmode='group',
                       color_discrete_sequence=['#1f3b5c', '#6baed6'],
                       title='Quarterly Revenue vs Expenses')
style(fig_quarterly)

# 4. Return on Equity
fig_roe = px.line(df, x='Month-Year', y='Return on Equity %',
                  markers=True, title='Return on Equity %')
fig_roe.update_traces(line_color='#08306b', line_width=3,
                       marker=dict(color='#08306b', size=8,
                                   line=dict(width=1.5, color='white')))
style(fig_roe)

# 5. NPL Ratio
fig_npl = px.line(df, x='Month-Year', y='NPL Ratio %',
                  markers=True, title='NPL Ratio % (Non Performing Loans)')
fig_npl.update_traces(line_color='#c0392b', line_width=3,
                       marker=dict(color='#c0392b', size=8,
                                   line=dict(width=1.5, color='white')))
style(fig_npl)

# 6. Total Assets
fig_assets = px.area(df, x='Month-Year', y='Total Assets (₦)',
                     title='Total Assets Growth',
                     color_discrete_sequence=['#2171b5'])
fig_assets.update_traces(line_color='#1f3b5c', line_width=3,
                          fillcolor='rgba(33,113,181,0.15)')
style(fig_assets)

# ── STYLES ────────────────────────────────────────────────────────────────────
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE,
    'padding': '16px 22px',
    'borderRadius': '12px',
    'textAlign': 'center',
    'flex': '1',
    'margin': '6px',
    'border': f'1.5px solid {BLUE_LIGHT}',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE,
    'padding': '10px',
    'borderRadius': '12px',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
    'flex': '1',
    'border': f'1px solid {BLUE_LIGHT}'
}
sidebar_label = {
    'backgroundColor': BLUE_MID,
    'color': WHITE,
    'padding': '7px 12px',
    'borderRadius': '6px',
    'marginBottom': '7px',
    'fontSize': '13px',
    'fontWeight': 'bold',
    'textAlign': 'center'
}

# ── APP LAYOUT ────────────────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H2('ZENITH BANK PLC — FINANCIAL REPORT DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '20px', 'letterSpacing': '0.5px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '16px 22px',
                       'borderRadius': '12px', 'flex': '2', 'marginRight': '12px'}
            ),
            html.Div([
                html.Div([
                    html.P('TOTAL REVENUE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold','letterSpacing':'0.5px'}),
                    html.H3(f'₦{total_revenue/1e9:,.1f}B', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('NET PROFIT', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold','letterSpacing':'0.5px'}),
                    html.H3(f'₦{total_profit/1e9:,.1f}B', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('AVG ROE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold','letterSpacing':'0.5px'}),
                    html.H3(f'{avg_roe:.1f}%', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('AVG NPL RATIO', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold','letterSpacing':'0.5px'}),
                    html.H3(f'{avg_npl:.1f}%', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '16px'}),

        # Body
        html.Div([

            # Sidebar
            html.Div([
                html.P('Years', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(str(y), style=sidebar_label) for y in [2022, 2023, 2024]],
                html.Br(),
                html.P('Quarters', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(q, style=sidebar_label) for q in ['Q1','Q2','Q3','Q4']],
                html.Br(),
                html.P('Metrics', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(m, style=sidebar_label) for m in ['Revenue','Profit','ROE','NPL']],
                html.Br(), html.Br(),
                html.P('Prepared by', style={'fontSize':'12px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'13px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '160px',
                'minWidth': '160px',
                'backgroundColor': WHITE,
                'padding': '16px',
                'borderRadius': '12px',
                'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
                'marginRight': '12px',
                'border': f'1px solid {BLUE_LIGHT}'
            }),

            # Charts — 2 per row for breathing room
            html.Div([
                # Row 1
                html.Div([
                    html.Div(dcc.Graph(figure=fig_revenue,   config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_profit,    config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                # Row 2
                html.Div([
                    html.Div(dcc.Graph(figure=fig_quarterly, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_roe,       config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                # Row 3
                html.Div([
                    html.Div(dcc.Graph(figure=fig_npl,    config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_assets, config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px'}),

            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8054)